VISUALIZATION

In [2]:
pip  install  plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.9 MB 10.1 MB/s eta 0:00:01
   --------- ------------------------------ 2.4/9.9 MB 7.1 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.9 MB 5.4 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.9 MB 4.8 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/9.9 MB 4.4 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.9 MB 4.0 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.9 MB 3.8 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.9 MB 3.6 MB/s eta 0:00:02
   ------------------------- -------------- 6.3/9.9 MB 3.5 MB/s eta 0:00:02
   --------------------------- ------------ 6.8/9.9 MB 3.4 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/9.9 MB 3.4 MB/s eta 0:00:01
   -------------------------------- ------- 8.1/9.9 MB 3.4 MB/s eta 0:00:01
   ---------------


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""
FULL VISUALIZATION SCRIPT
(Matplotlib + Seaborn + Plotly)
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import numpy as np
import os
from pathlib import Path
import itertools

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CSV_FILES = {
    "HLL":      r"E:\KEC TASK\Download data\HLL Historical Data.csv",
    "TCS":      r"E:\KEC TASK\Download data\TCS Historical Data.csv",
    "RELIANCE": r"E:\KEC TASK\Download data\RELI Historical Data.csv",
    "MSFT":     r"E:\KEC TASK\Download data\MSFT Historical Data.csv",
    "HDBK":     r"E:\KEC TASK\Download data\HDBK Historical Data (1).csv",
}

OUTPUT_DIR = "output_for_visuals"
os.makedirs(OUTPUT_DIR, exist_ok=True)

colors = itertools.cycle(["blue", "red", "green", "purple", "orange"])

# ─────────────────────────────────────────────
# LOAD + CLEAN
# ─────────────────────────────────────────────
def load_stock(path, ticker):
    df = pd.read_csv(path)
    df.columns = [c.strip().strip('"') for c in df.columns]

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    df = df.sort_values("Date")

    df["Price"] = df["Price"].astype(str).str.replace(",", "").astype(float)

    df["Return"] = df["Price"].pct_change()
    df["Volatility"] = df["Return"].rolling(10).std()

    df["EMA5"] = df["Price"].ewm(span=5).mean()
    df["EMA10"] = df["Price"].ewm(span=10).mean()

    df["Signal"] = np.where(df["EMA5"] > df["EMA10"], 1, 0)
    df["Crossover"] = df["Signal"].diff()

    return df


stocks = {}
for t, p in CSV_FILES.items():
    if Path(p).exists():
        stocks[t] = load_stock(p, t)

# ─────────────────────────────────────────────
# 1️⃣ MATPLOTLIB — PRICE TREND
# ─────────────────────────────────────────────
plt.figure(figsize=(12, 6))

for (t, df), c in zip(stocks.items(), colors):
    plt.plot(df["Date"], df["Price"], label=t, color=c)

plt.title("Stock Price Trend")
plt.legend()
plt.grid()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "matplotlib_price.png"))
plt.close()

# ─────────────────────────────────────────────
# 2️⃣ MATPLOTLIB — VOLATILITY
# ─────────────────────────────────────────────
plt.figure(figsize=(12, 6))

for (t, df), c in zip(stocks.items(), colors):
    plt.plot(df["Date"], df["Volatility"], label=t, color=c)

plt.title("Volatility Trend")
plt.legend()
plt.grid()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "matplotlib_volatility.png"))
plt.close()

# ─────────────────────────────────────────────
# 3️⃣ SEABORN — CORRELATION HEATMAP
# ─────────────────────────────────────────────
price_df = pd.DataFrame()

for t, df in stocks.items():
    temp = df[["Date", "Price"]].set_index("Date").rename(columns={"Price": t})
    price_df = temp if price_df.empty else price_df.join(temp, how="outer")

price_df = price_df.fillna(method="ffill")

corr = price_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap (Seaborn)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "seaborn_heatmap.png"))
plt.close()

# ─────────────────────────────────────────────
# 4️⃣ PLOTLY — INTERACTIVE PRICE + SIGNALS
# ─────────────────────────────────────────────
for t, df in stocks.items():

    fig = go.Figure()

    # Price
    fig.add_trace(go.Scatter(
        x=df["Date"], y=df["Price"],
        mode="lines", name="Price"
    ))

    # EMA
    fig.add_trace(go.Scatter(
        x=df["Date"], y=df["EMA5"],
        mode="lines", name="EMA5"
    ))

    fig.add_trace(go.Scatter(
        x=df["Date"], y=df["EMA10"],
        mode="lines", name="EMA10"
    ))

    # Buy/Sell
    buy = df[df["Crossover"] == 1]
    sell = df[df["Crossover"] == -1]

    fig.add_trace(go.Scatter(
        x=buy["Date"], y=buy["Price"],
        mode="markers", name="BUY",
        marker=dict(symbol="triangle-up", size=10)
    ))

    fig.add_trace(go.Scatter(
        x=sell["Date"], y=sell["Price"],
        mode="markers", name="SELL",
        marker=dict(symbol="triangle-down", size=10)
    ))

    fig.update_layout(
        title=f"{t} Interactive Chart",
        xaxis_title="Date",
        yaxis_title="Price"
    )

    fig.write_html(os.path.join(OUTPUT_DIR, f"{t}_interactive.html"))

print("\n✅ DONE — All Visualizations Generated!")

C:\Users\skitd\AppData\Local\Temp\ipykernel_18900\1868762923.py:101: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  price_df = price_df.fillna(method="ffill")



✅ DONE — All Visualizations Generated!
